# Exercise - Adversarial images and attacks with Keras

In [ ]:
# Download the code/images zip file
!unzip -qq basic-image-adversaries.zip
%cd basic-image-adversaries

In [ ]:
# [Optional] Install two required packages
# !pip install imutils
# !pip install opencv-python

In [ ]:
# import necessary packages
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.applications.resnet50 import decode_predictions
from tensorflow.keras.applications.resnet50 import preprocess_input
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import argparse
import imutils #pip install imutils
import json
import cv2 #pip install opencv-python
import os

### Stage 01: Normal image classification without adversarial attacks using Keras and TensorFlow

### Helper Methods [Given]
The following helper functions are provided for you:

#### 1. Image Display Function
- This function converts images from BGR (OpenCV format) to RGB (matplotlib format) and displays them with a title.
- OpenCV reads images in BGR color order, but matplotlib displays them in RGB order, so conversion is necessary for proper visualization.

In [ ]:
def plt_imshow(title, image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.imshow(image)
    plt.title(title)
    plt.grid(False)
    plt.show()

#### 2.ImageNet Class Index Helper
- This function maps human-readable class names (like "pig") to their numerical indices used by the model
- It loads a JSON file containing mappings between class names and indices
- Returns the index corresponding to the requested label, or None if not found
- This mapping is essential because the neural network works with numeric indices internally

In [ ]:
def get_class_idx(label):
    labelPath = os.path.join("pyimagesearch",
        "imagenet_class_index.json") #build the path to the ImageNet class label mappings file
    with open(labelPath) as f:
        imageNetClasses = {labels[1]: int(idx) for (idx, labels) in
            json.load(f).items()}   
    return imageNetClasses.get(label, None)
    # Open ImageNet mappings file and create a dictionary of {label: index} pairs
    # Return the index for the requested label, or None if not found

#### 3. Image Preprocessing Function
- Convert the image from BGR to RGB - neural networks are typically trained on RGB images
- Apply ResNet50-specific preprocessing (preprocess_input) - normalizes pixel values as expected by the model
- Resize the image to 224×224 pixels - this is the exact input size ResNet50 expects
- Add a batch dimension - neural networks expect inputs in batches, so we reshape from (224,224,3) to (1,224,224,3)

In [ ]:
def preprocess_image(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = preprocess_input(image)
    image = cv2.resize(image, (224, 224))
    image = np.expand_dims(image, axis=0)
    return image #return the preprocessed image

### Loading the Image
This cell replaces command-line argument parsing with a fixed dictionary because the code is running inside a Jupyter notebook.
Just read it as general knowledge in different use cases

In [ ]:
# [argparse code] Construct the argument parser and parse the arguments
# ap = argparse.ArgumentParser() # Construct the argument parser 
# ap.add_argument("-i", "--image", required=True, help="path to input image") # Defines an argument
# args = vars(ap.parse_args()) # Parse the arguments into a dictionary

# In a notebook, you are not usually running the file from a terminal with command-line arguments
# Instead, you can simply define the arguments as variables

In [ ]:
# Since we are using Jupyter Notebooks we can replace our argparse code with *hard coded* arguments and values
args = {
    "image": "pig.jpg" 
} 

In [ ]:
# [TODO] Load image from disk and make a clone for annotation
print("[INFO] loading image...")

# Your code should:
# 1. Use cv2.imread() to load args["image"]
# 2. Create a copy for visualization (so we don't modify the original)
# 3. Resize the output copy to width=400 using imutils.resize()
# 4. Call preprocess_image() on the original image to make it compatible with the neural network

### Loading the Model and Making Predictions

ResNet50 is a deep convolutional neural network (CNN) with 50 layers, commonly used for image classification and feature extraction.

It is pre-trained on ImageNet, a large-scale datasets which contains over a million images and 1,000 object categories.

In this exercise, it is the model being used to classify the image and later to evaluate adversarial examples

In [ ]:
# [GIVEN] Load the pre-trained ResNet50 model
print("[INFO] loading pre-trained ResNet50 model...")
model = ResNet50(weights="imagenet") #"weights=imagenet" specifies to use weights pre-trained on the ImageNet dataset

In [ ]:
# [TODO] Make predictions on the input image and parse the top-3 predictions
print("[INFO] making predictions...")

# Your code should:
# 1. Use model.predict() on preprocessedImage
# 2. Use decode_predictions() to convert results to labels, get the top 3 predictions with top=3 and first batch with [0]


### Displaying Predictions

In [ ]:
# [TODO] Write a for loop over the top three predictions
# Hints: 
# 1. Loop through each prediction tuple (containing imagenetID, label, probability)
# 2. For the top prediction (index 0), print both the label and its numerical class index
#    using get_class_idx() - we'll need this index for our adversarial attack
# 3. For all predictions, print their rank, label name, and confidence percentage
# 4. This helps us understand how the model classifies the original image

### Visualizing Results

In [ ]:
# [TODO] Draw the top-most predicted label on the image along with the confidence score. Finally, show the output image.
# Hints:
# 1. Create a text string with the top prediction and its confidence
# 2. Draw this text on the output image
# 3. Display the annotated image

---

### Stage 2: Adversarial attack 

### Preprocessing Function (Updated)
This version is slightly different from the previous one (see above) - it only handles color conversion, resizing, and adding a batch dimension, but skips normalization. This is intentional for the adversarial attack process.

In [ ]:
def preprocess_image(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224, 224))
    image = np.expand_dims(image, axis=0)
    return image

### Clipping Function

In [ ]:
# [TODO] define a function that clip the values of the tensor to a given range and return it

### Adversarial Attack Function

This cell defines the adversarial attack loop. Its job is to slightly modify the input image so the ResNet50 model starts making the wrong prediction

In [ ]:
# [TODO] Write a method (called generate_adversaries) which takes as input the pre-trained ResNet50 model, 
# main input images with 50 steps of iteration as training.
# Hints: 1. During the iterations  record our gradients explicitly indicate that our perturbation vector should
#           be tracked for gradient updates.
#        2. Add perturbation vector to the base image and preprocess the resulting image
#        3. Run this newly constructed image tensor through our model and calculate the loss 
#           with respect to the *original* class index
#        4. Check to see if we are logging the loss value, and if so, display it to our terminal
#        5. Calculate the gradients of loss with respect to the perturbation vector
#        6. Update the weights, clip the perturbation vector, and update its value
#        7. Return the perturbation vector

### Setting Parameters

In [ ]:
# [Given] construct the argument parser and parse the arguments
# ap = argparse.ArgumentParser()
# ap.add_argument("-i", "--input", required=True,
# help="path to original input image")
# ap.add_argument("-o", "--output", required=True,
# help="path to output adversarial image")
# ap.add_argument("-c", "--class-idx", type=int, required=True,
# help="ImageNet class ID of the predicted label")
# args = vars(ap.parse_args())

# since we are using Jupyter Notebooks we can replace our argument
# parsing code with *hard coded* arguments and values
args = {
    "input": "pig.jpg", #Input image: the original image to attack
    "output": "adversarial.png", #Output path: where to save the adversarial example
    "class_idx": 341 #341 represents a specific class in ImageNet (wombat) that we want the model to misclassify as
}

In [ ]:
# [Given] Define the epsilon and learning rate constants
EPS = 2 / 255.0
LR = 0.1

In [ ]:
# [Given] Load the input image from disk and preprocess it
print("[INFO] loading image...")
image = cv2.imread(args["input"])
image = preprocess_image(image)

### Loading Image and Model

In [ ]:
# [Given] Load the pre-trained ResNet50 model for running inference
print("[INFO] loading pre-trained ResNet50 model...")
model = ResNet50(weights="imagenet")

In [ ]:
# [TODO] Initialize adam optimizer and 'SparseCategoricalCrossentropy' as loss function

### Creating Tensors for Attack

In [ ]:
# [TODO] Create a tensor based off of the input image and initialize the
# perturbation vector (we will update this vector via training)

### Generating the Adversarial Example

In [ ]:
# [TODO] Generate the perturbation vector to create an adversarial example
print("[INFO] generating perturbation...")


### Creating and Saving the Adversarial Image

In [ ]:
print("[INFO] creating adversarial example...")
# [TODO] create the adversarial example, swap color channels, and save the output image to disk
# Hints:
# 1. Add the optimized perturbation to the base image
# 2. Convert from tensor to numpy array and remove the batch dimension
# 3. Clip pixel values to ensure they stay in valid range (0-255)
# 4. Convert back to BGR for OpenCV processing


### Running Inference on the Adversarial Image

In [ ]:
# [TODO] run inference with this adversarial example, parse the results,
# and display the top-1 predicted result
print("[INFO] running inference on the adversarial example...")

### Visualizing the Adversarial Example

In [ ]:
# [TODO] Draw the top-most predicted label on the adversarial image along
# with the confidence score and show the output image